In [ ]:
import sys
sys.path.append('/opt/ros/noetic/lib/python3/dist-packages/')
sys.path.append('/usr/lib/python3/dist-packages')
import numpy as np
from lab_utils.plan_utils import *
from matplotlib import pyplot as plt
from matplotlib import image as mpimg
import cv2
import rospy
from std_msgs.msg import Float32MultiArray
from jackal_msgs.msg import Drive
from geometry_msgs.msg import Pose, PoseStamped, PoseWithCovarianceStamped
from nav_msgs.msg import Odometry
from time import sleep
from scipy.spatial.transform import Rotation as R
from ipywidgets import Image, HBox, HTML
from ipyevents import Event
from IPython.display import clear_output, display  
from apriltag_ros.msg import AprilTagDetectionArray
from lab_utils.astart import AStarPlanner

from sensor_msgs.msg import Imu
from sensor_msgs.msg import LaserScan
from sensor_msgs.msg import CompressedImage, Image as Img
from mobile_manip.srv import ReachName, GetValues, ReachValues

################################################################################
# 1) interface existante (boutons, carte cliquable, etc.)
################################################################################
from ipywidgets import HBox, Box, Layout, GridspecLayout, Image
import ipywidgets as widgets
import threading
import time
from ipyevents import Event  # Pour gérer les événements de clic
import io
import numpy as np
from matplotlib import pyplot as plt
from collections import namedtuple

from os import environ

sim = True


In [ ]:
################################################################################
# Obtenir l'adresse IP locale de la machine sur le réseau
################################################################################

import socket
def get_ip():
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    s.settimeout(0)
    try:
        s.connect(('192.168.0.254', 1))
        IP = s.getsockname()[0]
    except Exception:
        IP = '127.0.0.1'
    finally:
        s.close()
    return IP
print(get_ip())

In [ ]:
################################################################################
# Callback AMCL : Mise à jour et publication de la pose estimée du robot
################################################################################
if sim :
    # AMCL_pose callback (pour la visualisation de la pose)
    pose_msg = Pose()
    def amcl_callback(msg):
        global pose_msg
        pose_msg = msg.pose.pose
        pose = PoseStamped()

        pose.header.seq = 1
        pose.header.stamp = rospy.Time.now()
        pose.header.frame_id = "map"

        pose.pose.position.x = pose_msg.position.x
        pose.pose.position.y = pose_msg.position.y
        pose.pose.position.z = 0.0

        pose.pose.orientation.x = pose_msg.orientation.x
        pose.pose.orientation.y = pose_msg.orientation.y
        pose.pose.orientation.z = pose_msg.orientation.z
        pose.pose.orientation.w = pose_msg.orientation.w

        pose_pub.publish(pose)

In [ ]:
################################################################################
# Laser scan subscriber callback
################################################################################

laser_msg = LaserScan()
def laser_scan_callback(msg):
    global laser_msg
    laser_msg = msg

In [ ]:
################################################################################
# Odom_pose callback (pour la visualisation de la pose)
################################################################################
if sim:
    pose_msg = Pose()
    #fonction pour la Simulation
    def odom_callback(msg):
        global pose_msg
        pose_msg = msg.pose.pose

    odom_sub = rospy.Subscriber('/odometry/filtered', Odometry, odom_callback)

In [ ]:
################################################################################
# Realsense Pose subscriber callback 
################################################################################

if not(sim):
    pose_msg = Odometry() # diff avec l'ancienne version : pose_msg = Pose() tout le temps

# Fonction pour la réalité
def pose_callback(msg):
    global pose_msg
    pose_msg  = msg.pose.pose

# Fonction pour le calcul de l'orientation à partir d'un quaternion
def get_heading_from_quaternion(q):
    r = R.from_quat([q.x, q.y, q.z, q.w])
    angles = r.as_euler('xyz', degrees=False)
    return angles[2]

# Normalise un angle dans l'intervalle [-π, π] (en radians)
def wraptopi(angle):
    xwrap=np.remainder(angle, 2*np.pi)
    if np.abs(xwrap)>np.pi:
        xwrap -= 2*np.pi * np.sign(xwrap)
    return xwrap

In [ ]:
################################################################################
# Imu subscriber callback
################################################################################
imu_msg = Imu()
def imu_callback(msg):
    global imu_msg
    imu_msg = msg

In [ ]:
################################################################################
# Camera subscriber callback
################################################################################
cam_msg = CompressedImage()
def cam_callback(msg):
    global cam_msg
    cam_msg = msg

In [ ]:
# ROS subscribers et publishers
laser_scan_sub = rospy.Subscriber('/scan', LaserScan, laser_scan_callback)
#pose_sub = rospy.Subscriber('/mobile_manip/t265/odom/sample', Odometry, pose_callback)
#cmd_drive_pub = rospy.Publisher('/mobile_manip/dingo_velocity_controller/cmd_drive', Drive, queue_size=1)

In [ ]:
tag_array = AprilTagDetectionArray()
def tag_callback(msg):
    global tag_array
    tag_array  = msg


In [ ]:
if sim:
    pass
else:
    environ['ROS_MASTER_URI'] = "http://cpr-ets05-04.local:11311/"
    environ['ROS_IP'] = get_ip()

if sim:
# ROS subscribers et publishers
    cmd_drive_pub = rospy.Publisher('/mobile_manip/dingo_velocity_controller/cmd_drive', Drive, queue_size=1)
    pose_pub = rospy.Publisher('/mobile_manip/pose', PoseStamped, queue_size=1)
    pose_sub = rospy.Subscriber('/mobile_manip/pose', Odometry, queue_size=1)#on l'a rajouté
    amcl_sub = rospy.Subscriber('/amcl_pose', PoseWithCovarianceStamped, amcl_callback)
    #pose_sub = rospy.Subscriber('/mobile_manip/dingo_velocity_controller/odom', Odometry, odom_callback)
    #pose_sub = rospy.Subscriber('/mobile_manip/t265/odom/sample', Odometry, pose_callback)
    imu_sub = rospy.Subscriber('/imu/data', Imu, imu_callback)
    cam_sub = rospy.Subscriber('/mobile_manip/d435i/color/image_raw/compressed', CompressedImage, cam_callback)
    tag_sub = rospy.Subscriber('/mobile_manip/tag_detections', AprilTagDetectionArray, tag_callback)

else:
    cmd_drive_pub = rospy.Publisher('/mobile_manip/base/dingo_velocity_controller/cmd_drive', Drive, queue_size=1)
    pose_sub = rospy.Subscriber('/odometry/filtered', Odometry,pose_callback)
    #amcl_sub = rospy.Subscriber('/mobile_manip/base/amcl_pose', PoseWithCovarianceStamped, amcl_callback)
    #wheel_odom_sub = rospy.Subscriber('/mobile_manip/base/dingo_velocity_controller/odom', Odometry, wheel_pose_callback)
    cam_sub = rospy.Subscriber('/mobile_manip/d435i/color/image_raw/compressed', CompressedImage, cam_callback)
    imu_sub = rospy.Subscriber('/mobile_manip/base/imu/data', Imu, imu_callback)
    tag_sub = rospy.Subscriber('/mobile_manip/tag_detections', AprilTagDetectionArray, tag_callback)
    
    # dans ancienne version pour la sim :
    #tag_sub = rospy.Subscriber('/tag_detections', AprilTagDetectionArray, tag_callback)

In [ ]:
rospy.init_node('dingo_controller', anonymous=True)
start = float(rospy.Time().now().secs)
rate = rospy.Rate(50)

cmd_drive_pub = rospy.Publisher('/mobile_manip/base/dingo_velocity_controller/cmd_drive', Drive, queue_size=1)

interwheel_distance = 0.46
left_wheel_radius = 0.045
right_wheel_radius = 0.045
def move_robot(linear, angular):
    vel_left  = (linear - angular * interwheel_distance / 2.0) / left_wheel_radius
    vel_right = (linear + angular * interwheel_distance / 2.0) / right_wheel_radius

    # Envoi des commandes au roues par topic ROS
    cmd_drive_msg = Drive()
    cmd_drive_msg.drivers[0] = vel_left
    cmd_drive_msg.drivers[1] = vel_right
    cmd_drive_pub.publish(cmd_drive_msg)

In [ ]:
def reach_coord(coord):
    effector_goal = [float(coord[0]), float(coord[1]), float(coord[2]), 0, 90, 0, 0]
    print(effector_goal)
    rospy.wait_for_service('/mobile_manip/reach_cartesian')
    reach_pose = rospy.ServiceProxy('/mobile_manip/reach_cartesian', ReachValues)
    reach_pose(effector_goal)

def reach_recorded(state):
    rospy.wait_for_service('/mobile_manip/reach_name')
    reach_recorded_pose = rospy.ServiceProxy('/mobile_manip/reach_name', ReachName)
    reach_recorded_pose(state)

In [ ]:
log_output = widgets.Output(
    layout={'height': '60px', 'width': '85%', 'border': '1px solid black', 'overflow': 'auto'}
)
def log_action(msg):
    with log_output:
        print(" " + msg)

In [ ]:
def get_heading_from_quaternion(q):
    if(np.sqrt(q.x**2+q.y**2+q.z**2+q.w**2) == 0):
        print('Quaternion norm 0 : ', q.x, q.y, q.z, q.w)
        return 0
    else:
        r = R.from_quat([q.x, q.y, q.z, q.w])
        angles = r.as_euler('xyz', degrees=False)
        return angles[2]


In [ ]:
###################################################################################################
# Planification de trajectoire avec A* et conversion des coordonnées (map → monde réel)
###################################################################################################
def trajectoire (map, start, end):
    try:
        astarPlanner = AStarPlanner(map=map, step_size=10, collision_radius=20, heuristic_dist='Euclidean')
        astarPlanner.plan(start=start, target=end)
    except ValueError:
        log_action("aucun chemin trouvé")
        return [tuple([pose_msg.position.x, pose_msg.position.y])], []
    traj = []
    #offset = 0.1
    offset = 0 ##if not sim

    fig = plt.figure(figsize=(16,5))
    ax = fig.add_subplot(1, 2, 1)
    #plt.imshow(image)
    ax.add_artist(plt.Circle((start.x, start.y), 10, color='r'))
    ax.add_artist(plt.Circle((end.x, end.y), 10, color='y'))
    for i in range(len(astarPlanner.open_list)-1):
        pt = astarPlanner.open_list[i].pos.tuple()
        ax.add_artist(plt.Circle((pt[0], pt[1]), 5, color='gray'))
    for i in range(len(astarPlanner.close_list)-1):
        pt = astarPlanner.close_list[i].pos.tuple()
        ax.add_artist(plt.Circle((pt[0], pt[1]), 5, color='gray'))

    ax2 = fig.add_subplot(1, 2, 2)
    #plt.imshow(image)
    ax2.add_artist(plt.Circle((start.x, start.y), 10, color='r'))
    ax2.add_artist(plt.Circle((end.x, end.y), 10, color='y'))
    for i in range(len(astarPlanner.finalPath)-1):
        pt = astarPlanner.finalPath[i].tuple()
        ax2.add_artist(plt.Circle((pt[0], pt[1]), 5, color='g'))
        traj.append(astarPlanner.finalPath[i].tuple())
    #plt.show()

    for i in traj:
        temp = list(i)  
        temp[0] -= 1444.7
        temp[1] -= 1052.1
        temp[0] *= 1/50
        temp[1] *= -1/50

        temp[0] -= offset

        traj[traj.index(i)] = tuple(temp) 

    log_action(f"Trajectoire calculée : {traj} (N={len(traj)})")
    return traj, astarPlanner.finalPath


In [ ]:
##################################################################################################
# Commande de déplacement vers une position cible avec contrôle d'orientation et arrêt d'urgence
# Algorithme de navigation vers un point : ajuste l'orientation puis avance en ligne droite
##################################################################################################
def goto (pos):
    global stop_requested

    err_x = pos[0] - pose_msg.position.x
    err_y = pos[1] - pose_msg.position.y
    h = np.sqrt(err_x*err_x + err_y*err_y)
    err_angle = np.arctan2(err_y/h, err_x/h) - get_heading_from_quaternion(pose_msg.orientation)

    
    while (np.abs(err_angle)>0.1):
        if stop_requested:
            move_robot(0, 0)
            log_action("Arrêt pendant rotation.")
            return
         
        command = 0.2*(err_angle/np.abs(err_angle))
        move_robot(0, command)
        sleep(0.01)
        err_angle = np.arctan2(err_y/h, err_x/h) - get_heading_from_quaternion(pose_msg.orientation)

    while (np.abs(err_x)>0.1 or np.abs(err_y)>0.1):
        if stop_requested:
            move_robot(0, 0)
            log_action("Arrêt pendant déplacement.")
            return
        
        move_robot(0.1,0)
        sleep(0.01)
        err_x = pos[0] - pose_msg.position.x
        err_y = pos[1] - pose_msg.position.y
        h = np.sqrt(err_x*err_x + err_y*err_y)
    move_robot(0,0)
    return



In [ ]:
###############################################################################################
# Transformation des coordonnées du tag depuis le repère caméra vers le repère monde
###############################################################################################
def r2w (tag_array):
    T = np.array([[0,0,1,0.1852], [-1,0,0,0.024896], [0,-1,0,0.45805], [0, 0, 0, 1]])
    theta = get_heading_from_quaternion(pose_msg.orientation)
    t = [[np.cos(theta), -np.sin(theta), 0, pose_msg.position.x], [np.sin(theta), np.cos(theta), 0, pose_msg.position.y], [0, 0, 1, pose_msg.position.z], [0, 0, 0, 1]]
    x = tag_array.detections[0].pose.pose.pose.position.x
    y = tag_array.detections[0].pose.pose.pose.position.y
    z = tag_array.detections[0].pose.pose.pose.position.z
    temp = np.array([[x],[y],[z],[1]])
    return np.dot(np.dot(t,T),temp)



In [ ]:
####################################################################################
# Vérifie si un tag détecté est déjà présent dans la liste des tags visités
####################################################################################
def tag_visite(tag_list):
    for tag in tag_list :
        if ((np.abs(tag[0]-r2w(tag_array)[0])<1) and (np.abs(tag[1]-r2w(tag_array)[1])<1)):
            return True
    return False

In [ ]:
###############################################################################
# Exécute une trajectoire avec détection et traitement des tags AprilTag
###############################################################################
def suivre(traj, tag, tag_list):
    global stop_requested
    if stop_requested:
        move_robot(0, 0)
        log_action("Trajectoire interrompue avant démarrage.")
        return
    stop_requested = False  # on reset l'état à chaque appel de trajectoire

    for i in traj:
        if stop_requested:
            move_robot(0, 0)
            with log_output:
                print("Trajectoire interrompue par l'utilisateur")
            return
        if (tag_array.detections != []) and not(tag) and not(tag_visite(tag_list)):
            with log_output:
                print("Tag detecté")
            goal = r2w(tag_array)
            with log_output:
                print("Tag :" + np.array2string(goal, precision=4, suppress_small=True))

            log_action(f"x={pose_msg.position.x}, y={pose_msg.position.y}")

            tag_list.append(goal)
            if (pose_msg.position.x<goal[0]):
                goal = Point((goal[0]-0.7)*50+1444.7, (goal[1])*(-50)+1052.1)
            else:
                goal = Point((goal[0]+0.7)*50+1444.7, (goal[1])*(-50)+1052.1)
            
            pos = Point(pose_msg.position.x*50+1444.7, pose_msg.position.y*(-50)+1052.1)
            end = Point(traj[-1][0]*50+1444.7, traj[-1][1]*(-50)+1052.1)
            traj_tag, path_nodes_tag = trajectoire(map, pos, goal)

            if len(path_nodes_tag) == 0:
                log_action("Impossible de rejoindre le tag, on reprend la trajectoire initiale")
            else:
                if len(traj_tag) > 2:
                    for _ in range(2):
                        traj_tag.pop(-1)
                        if stop_requested:
                            move_robot(0, 0)
                            log_action("Arrêt pendant suivi tag")
                            return
                suivre(traj_tag, True, tag_list)

                #faire bouger le bras
            pos = Point(pose_msg.position.x*50+1444.7, pose_msg.position.y*(-50)+1052.1)
            traj, path_nodes  = trajectoire(map, pos,end)
            if stop_requested:
                move_robot(0, 0)
                log_action("Arrêt pendant retour à la trajectoire")
                return
            suivre(traj, False,tag_list)
            log_action("tag atteint")
            reach_coord([0.5,0,0])
            return
        if i[0]!=0 and i[1]!=0 :
            goto(i)
    move_robot(0,0)
    return


In [ ]:
####################################################################################
# Point d'entrée principal pour l'exécution de la navigation et la gestion des tags
####################################################################################
def main(traj):
    reach_recorded("vertical") # Rajouté pour mettre le bras vertical vers le haut
    tag_list = []
    suivre(traj,False,tag_list)
    return


In [ ]:
stop_requested = False

#-----------------------------------------------------------------------------------
##                          Cliquer sur la carte
#-----------------------------------------------------------------------------------


# Dummy pose 
pose_msg = type('', (), {})()
pose_msg.position = type('', (), {})()
pose_msg.position.x = 0
pose_msg.position.y = 0
pose_msg.orientation = namedtuple("Quaternion", "x y z w")(0, 0, 0, 1)



image_file = open("Maps/a2230_map_closed.png", 'rb')
image = mpimg.imread("Maps/a2230_map_closed.png")
height, width = image.shape[0], image.shape[1]


# Variable Gobales
start = None
end = None
click_coords = None
img_bytes = None

map_widget = widgets.Image(
        value=image_file.read(),
        # value=img_bytes,
        format="jpeg",
        height="30%",
        width="50%")

# Superposition (stack)
stack = widgets.Box(children=[map_widget], layout=widgets.Layout(position='relative'))
ui = widgets.VBox([stack, log_output])
# display(ui)

# Initialise la carte de navigation avec le point de départ et prépare l'affichage
def init_map():
        global img_bytes
        start = Point(1444.7, 1052.1)
        fig = plt.figure(figsize=(width / 100, height / 100), dpi=100)
        ax = fig.add_axes([0, 0, 1, 1]) 
        plt.imshow(image)
        ax.add_artist(plt.Circle((start.x, start.y), 10, color='r'))
        ax.axis('off')  # pour ne pas afficher les axes

        # Sauvegarder la figure dans un buffer
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight', pad_inches=0)
        buf.seek(0)
        img_bytes = buf.getvalue()
        map_widget.value = img_bytes
        buf.close()
        plt.close(fig)


# Fonction de mise à jour du chemin
def update_path():
    global start, map, stop_requested

    if click_coords is None:
        log_action("Aucun clic détecté.")
        return

    map_img = 1-np.array(image[:,:,1])
    mat_map = map_img
    map = BMPMap(width=map_img.shape[1], height=map_img.shape[0], mat=mat_map)

    start = Point(pose_msg.position.x*50+1444.7, pose_msg.position.y*(-50)+1052.1)
    end = Point(click_coords[0], click_coords[1])
    
    traj, path_nodes = trajectoire(map, start, end)
    traj_px = [node.tuple() for node in path_nodes]

    # Tracé
    global img_bytes
    fig = plt.figure(figsize=(width / 100, height / 100), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1]) 
    plt.imshow(image)

    ########## TRACE PTS DEP ET END EN ROUGE ET JAUNE #################
    ax.add_artist(plt.Circle((start.x, start.y), 10, color='r'))
    ax.add_artist(plt.Circle((end.x, end.y), 10, color='y'))
    ###################################################################

    ################## TRACE TRAJECTOIRE EN VERT ######################
    for i in traj_px:
        ax.add_artist(plt.Circle((i[0], i[1]), 5, color='g'))
    ax.axis('off')  # pour ne pas afficher les axes
    ###################################################################
    
    plt.show()

    # Sauvegarder la figure dans un buffer
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight', pad_inches=0)
    buf.seek(0)
    img_bytes = buf.getvalue()
    map_widget.value = img_bytes
    buf.close()
    plt.close(fig)
    
    stop_requested = False 

    # Lancement de la trajectoire dans un thread
    thread_traj = threading.Thread(target=main, args=(traj,))
    thread_traj.start()

# Clic sur image invisible au-dessus
def on_click_handler(event):
    if mode_selector.value != 'Auto':
        log_action("Clic ignoré : Passez en mode autonome pour interagir avec la carte.")
        return
    
    global click_coords
    click_coords = [event['dataX'], event['dataY']]
    log_action(f"Clic détecté brut : {click_coords}")
    update_path()

init_map()
click_event = Event(source=map_widget, watched_events=['click'])
click_event.on_dom_event(on_click_handler)
#-----------------------------------------------------------------------------------

#-----------------------------------------------------------------------------------
##                          Retour caméra (exemple)
#-----------------------------------------------------------------------------------
camera = widgets.Image(
    value=b'',  # valeur initiale vide
    format='jpeg',
    height="120%",
    width="200%"
)

# mise à jour continu du flux vidéo de la caméra
def update_camera():
    while True:
        try:
            if hasattr(cam_msg, 'data'):
                camera.value = cam_msg.data  # JPEG compressé
        except Exception as e:
            log_action(f"[Camera] Erreur : {e}")
        time.sleep(0.1)

#-----------------------------------------------------------------------------------
##                          Boutons de contrôle
#-----------------------------------------------------------------------------------
btn_up = widgets.Button(icon='arrow-up')
btn_left = widgets.Button(icon='arrow-left')
btn_down = widgets.Button(icon='arrow-down')
btn_right = widgets.Button(icon='arrow-right')
btn_stop = widgets.Button(description='Stop')

def on_btn_up_clicked(b):
    speed = speed_slider.value
    move_robot(speed, 0)
    log_action(f"Avancer à vitesse {speed}")

def on_btn_left_clicked(b):
    speed = speed_slider.value
    move_robot(0, speed)
    log_action(f"Tourner à gauche à vitesse {speed}")

def on_btn_down_clicked(b):
    speed = speed_slider.value
    move_robot(-speed, 0)
    log_action(f"Reculer à vitesse {speed}")

def on_btn_right_clicked(b):
    speed = speed_slider.value
    move_robot(0, -speed)
    log_action(f"Tourner à droite à vitesse {speed}")

def on_btn_stop_clicked(b):
    global stop_requested
    stop_requested = True
    move_robot(0, 0)
    log_action("Arrêt")


btn_up.on_click(on_btn_up_clicked)
btn_left.on_click(on_btn_left_clicked)
btn_down.on_click(on_btn_down_clicked)
btn_right.on_click(on_btn_right_clicked)
btn_stop.on_click(on_btn_stop_clicked)

#-----------------------------------------------------------------------------------
##                          Modes Manuel / Auto
#-----------------------------------------------------------------------------------
mode_selector = widgets.ToggleButtons(
    options=['Manuel', 'Auto'],
    description='Mode:',
    style={'description_width': 'initial'}
)

# Gestion du changement de mode (Manuel/Auto) et configuration de l'interface
def switch_mode(change):
    is_auto = (change['new'] == 'Auto')

    # Activation/désactivation des boutons selon le mode
    for btn in [btn_up, btn_down, btn_left, btn_right]:
        btn.disabled = is_auto

    btn_stop.disabled = False  # Toujours actif    
    speed_slider.disabled = is_auto

    if is_auto:
        map_widget.layout.pointer_events = 'auto'
        map_widget.layout.opacity = '1.0'
        log_action("Mode: Autonome")
    else:
        map_widget.layout.pointer_events = 'none'
        map_widget.layout.opacity = '0.5'
        log_action("Mode: Manuel")

mode_selector.observe(switch_mode, names='value')

#-----------------------------------------------------------------------------------
##                          Slider de vitesse
#-----------------------------------------------------------------------------------
speed_slider = widgets.FloatSlider(
    description='Vitesse', min=0.1, max=1.0, step=0.1, value=0.3 # Modifiée (avant 0.5)
)
spacer = widgets.Label(value="")

#-----------------------------------------------------------------------------------
##                          Mise en place globale
#-----------------------------------------------------------------------------------
ui = widgets.VBox([
    mode_selector,
    speed_slider,
    log_output,
    spacer
])

grid = GridspecLayout(8, 8, height='600px')
grid[0:4, 0:3] = camera   # Caméra
grid[0:4, 4:8] = map_widget  # Carte cliquable

grid[5, 1] = btn_up
grid[6, 0] = btn_left
grid[6, 1] = btn_stop
grid[6, 2] = btn_right
grid[7, 1] = btn_down

# Thread pour la caméra
thread_cam = threading.Thread(target=update_camera, daemon=True)
thread_cam.start()

display(ui)
with log_output:
    print("Test direct du widget log_output")

display(grid)



################################################################################
# 2) LIDAR rendu dans un widget Image (non-cliquable, grisé)
################################################################################
import io
import matplotlib
matplotlib.use('Agg')  # Dessin hors-écran (pas de figure interactive)
import matplotlib.pyplot as plt

# 1) Créer un widget "Image" grisé et non-cliquable
lidar_image = widgets.Image(
    format='png',
    layout=Layout(
        width='400px',        
        height='350px',       
        pointer_events='none',# Empêche tout clic
        opacity='0.5'         # Aspect « grisé »
    )
)

# 2) Positionner ce widget dans la grille sous la carte
grid[4:8, 4:8] = lidar_image

# 3) Thread pour dessiner régulièrement le LIDAR hors-écran et mettre à jour l'image
def update_lidar():
    while True:
        fig = matplotlib.figure.Figure(figsize=(6, 3))
        canvas = matplotlib.backends.backend_agg.FigureCanvasAgg(fig)
        ax = fig.add_subplot(111)

        # Charger l'image de fond si elle existe
        try:
            bg_img = matplotlib.image.imread("Images/Fond.png")
            ax.imshow(bg_img, extent=[-10, 10, -10, 10], zorder=0)
        except FileNotFoundError:
            pass

        scatter_obstacles = ax.scatter([], [], color='r')
        scatter_robot = ax.scatter([0], [0], color='lime')

        ax.set_xlim([-10, 10])
        ax.set_ylim([-10, 10])
        ax.grid(True)

        # -- Récupération des données LIDAR --
        if (pose_msg is not None) and ('laser_msg' in globals()) and (laser_msg is not None):
            cap_global = wraptopi(get_heading_from_quaternion(pose_msg.orientation))
            x_pos_global = pose_msg.position.x
            y_pos_global = pose_msg.position.y

            obstacles_xy_local = []
            for i, r in enumerate(laser_msg.ranges):
                angle_abs = cap_global + (laser_msg.angle_min + i * laser_msg.angle_increment) + math.pi/2
                abs_x = x_pos_global + r * math.cos(angle_abs)
                abs_y = y_pos_global + r * math.sin(angle_abs)

                dx = abs_x - x_pos_global
                dy = abs_y - y_pos_global

                sinC = math.sin(-cap_global)
                cosC = math.cos(-cap_global)
                x_loc = dx * cosC - dy * sinC
                y_loc = dx * sinC + dy * cosC
                obstacles_xy_local.append([x_loc, y_loc])

                if i == 0 and (0.01 < r < 0.1):
                    move_robot(0, 0)

            scatter_obstacles.set_offsets(obstacles_xy_local)
            scatter_robot.set_offsets([[0, 0]])

        # Dessiner explicitement avec canvas.draw()
        canvas.draw()

        # Sauvegarder la figure en PNG dans le buffer mémoire
        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight')
        buf.seek(0)

        # Mise à jour widget Image
        lidar_image.value = buf.getvalue()

        buf.close()
        plt.close(fig)

        time.sleep(0.1)

# 4) Démarrer ce thread
thread_lidar = threading.Thread(target=update_lidar, daemon=True)
thread_lidar.start()
